# Mutual Credit Velocity Analysis

This notebook analyzes transaction velocity over time to detect economic health and failure modes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## Load Simulation Results

Load metrics from one or more simulation runs.

In [ ]:
# Load baseline scenario
baseline_df = pd.read_csv('../results/baseline/metrics.csv')
baseline_df['Scenario'] = 'Baseline'

# Load high demurrage scenario
demurrage_df = pd.read_csv('../results/high_demurrage/metrics.csv')
demurrage_df['Scenario'] = 'High Demurrage'

# Combine
all_scenarios = pd.concat([baseline_df, demurrage_df], ignore_index=True)

print(f"Loaded {len(all_scenarios)} observations across {all_scenarios['Scenario'].nunique()} scenarios")
print(f"\nColumns: {list(all_scenarios.columns)}")

## Velocity Over Time

Plot transaction velocity to identify trends and collapse points.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for scenario in all_scenarios['Scenario'].unique():
    data = all_scenarios[all_scenarios['Scenario'] == scenario]
    ax.plot(data['Month'], data['Velocity'], label=scenario, linewidth=2, marker='o', markersize=4)

ax.axhline(y=300, color='red', linestyle='--', label='Collapse Threshold', alpha=0.5)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Monthly Transaction Volume (credits)', fontsize=12)
ax.set_title('Economic Velocity Over Time', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Inequality Metrics (Gini Coefficient)

Track how balance inequality evolves over time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gini over time
for scenario in all_scenarios['Scenario'].unique():
    data = all_scenarios[all_scenarios['Scenario'] == scenario]
    ax1.plot(data['Month'], data['Gini'], label=scenario, linewidth=2, marker='o', markersize=4)

ax1.axhline(y=0.7, color='red', linestyle='--', label='Extreme Inequality Threshold', alpha=0.5)
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Gini Coefficient', fontsize=12)
ax1.set_title('Balance Inequality Over Time', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Hoarding index over time
for scenario in all_scenarios['Scenario'].unique():
    data = all_scenarios[all_scenarios['Scenario'] == scenario]
    ax2.plot(data['Month'], data['HoardingIndex'], label=scenario, linewidth=2, marker='o', markersize=4)

ax2.axhline(y=0.8, color='red', linestyle='--', label='Hoarding Concentration Threshold', alpha=0.5)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Hoarding Index (Top 10% Share)', fontsize=12)
ax2.set_title('Hoarding Concentration Over Time', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Default Rate Analysis

Track the percentage of transactions that defaulted.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for scenario in all_scenarios['Scenario'].unique():
    data = all_scenarios[all_scenarios['Scenario'] == scenario]
    ax.plot(data['Month'], data['DefaultRate'] * 100, label=scenario, linewidth=2, marker='o', markersize=4)

ax.axhline(y=15, color='red', linestyle='--', label='System Stress Threshold (15%)', alpha=0.5)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Default Rate (%)', fontsize=12)
ax.set_title('Transaction Default Rate Over Time', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Trust Dynamics

Observe how average trust scores evolve.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for scenario in all_scenarios['Scenario'].unique():
    data = all_scenarios[all_scenarios['Scenario'] == scenario]
    ax.plot(data['Month'], data['AvgTrust'], label=scenario, linewidth=2, marker='o', markersize=4)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Average Trust Score', fontsize=12)
ax.set_title('Trust Network Health Over Time', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary Statistics

Compare final outcomes across scenarios.

In [ ]:
# Get final month data for each scenario
final_stats = all_scenarios.groupby('Scenario').last()[[
    'Velocity', 'TotalVolume', 'Transactions', 'DefaultRate', 
    'Gini', 'HoardingIndex', 'AgentsInDefault', 'AvgTrust'
]]

# Format for display
final_stats['DefaultRate'] = (final_stats['DefaultRate'] * 100).round(2)
final_stats['HoardingIndex'] = (final_stats['HoardingIndex'] * 100).round(2)
final_stats = final_stats.round(2)

# Rename columns for clarity
final_stats.columns = [
    'Velocity', 'Total Volume', 'Transactions', 'Default %', 
    'Gini', 'Hoarding %', 'In Default', 'Avg Trust'
]

print("Final Outcomes by Scenario:\n")
print(final_stats.to_string())

# Highlight best/worst performers
print("\n" + "="*80)
print("Key Findings:")
print("="*80)
print(f"Highest Velocity: {final_stats['Velocity'].idxmax()} ({final_stats['Velocity'].max():.1f} credits/month)")
print(f"Lowest Default Rate: {final_stats['Default %'].idxmin()} ({final_stats['Default %'].min():.2f}%)")
print(f"Most Equal (lowest Gini): {final_stats['Gini'].idxmin()} ({final_stats['Gini'].min():.2f})")
print(f"Least Hoarding: {final_stats['Hoarding %'].idxmin()} ({final_stats['Hoarding %'].min():.2f}%)")

## Agent-Level Analysis

Look at individual agent outcomes by type.

In [ ]:
# Load agent data for baseline scenario
agents_df = pd.read_csv('../results/baseline/agents.csv')

# Get final state for each agent
final_agents = agents_df.groupby('AgentID').last().reset_index()

# Plot balance distribution by agent type
fig, ax = plt.subplots(figsize=(12, 6))

agent_types = final_agents['Type'].unique()
positions = range(len(agent_types))

balance_by_type = [final_agents[final_agents['Type'] == t]['Balance'].values for t in agent_types]

bp = ax.boxplot(balance_by_type, positions=positions, labels=agent_types, patch_artist=True)

# Color the boxes
colors = plt.cm.Set3(range(len(agent_types)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Agent Type', fontsize=12)
ax.set_ylabel('Final Balance (credits)', fontsize=12)
ax.set_title('Balance Distribution by Agent Type (Baseline Scenario)', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Print summary stats by type
print("\nBalance Statistics by Agent Type:\n")
print(final_agents.groupby('Type')['Balance'].describe().round(2))

## Conclusions

Summarize findings and recommendations for ICN's economic parameters.

In [ ]:
print("Simulation Conclusions:")
print("=" * 80)
print("\n1. Does demurrage prevent hoarding?")
print("   - Compare baseline vs high_demurrage hoarding indices")
print("\n2. Do dynamic credit limits reduce defaults?")
print("   - Compare baseline vs dynamic_limits default rates")
print("\n3. What's the system's free-rider tolerance?")
print("   - Check high_free_riders scenario for collapse threshold")
print("\n4. Can the economy function in low-trust environments?")
print("   - Analyze low_trust velocity and fragmentation")
print("\nNext Steps:")
print("- Validate against pilot community data (Track C2)")
print("- Refine parameter recommendations based on real-world feedback")
print("- Test additional scenarios (external shocks, scale-up, etc.)")